### Configuration, inputs & outputs

In [1]:
import subprocess
from pathlib import Path

from pipeline_utils import make_case3_paths

paths = make_case3_paths()

nest_vnn_dir = paths["project"] / "ml_models" / "nest_vnn"
nest_python = nest_vnn_dir / "nest_env" / "bin" / "python"

# Pretrained cisplatin ensemble (5-fold) -- same model applied to all cohorts below
PRETRAINED_MODEL = lambda fold: nest_vnn_dir / "pretrained_models" / "cisplatin" / f"model_cisplatin_{fold}.pt"
STD_FILE = nest_vnn_dir / "sample" / "std.txt"
N_FOLDS = 5

# Cohorts: point directly at case3's own NeST-VNN input dirs (no copying) --
# gene2ind.txt / file format already confirmed identical to what predict.py expects.
COHORTS = {
    "McGill_cisplatin": paths["proc"] / "McGill_nest_input",
    "UHN_carboplatin_complete": paths["proc"] / "UHN_nest_input" / "complete_cohort",
    "UHN_carboplatin_strict": paths["proc"] / "UHN_nest_input" / "strict_cohort",
}

hidden_root = nest_vnn_dir / "hidden"
results_root = nest_vnn_dir / "results"

### Run the pretrained 5-fold cisplatin ensemble on each cohort

In [2]:
def run_nest_vnn_predict(cohort_name, test_data_dir, n_folds=N_FOLDS):
    """Run the pretrained NeST-VNN cisplatin ensemble (predict.py) on one cohort's prepared inputs."""
    hidden_dir = hidden_root / cohort_name
    result_dir = results_root / cohort_name
    hidden_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    for fold in range(1, n_folds + 1):
        cmd = [
            str(nest_python), "src/predict.py",
            "-gene2id", str(test_data_dir / "gene2ind.txt"),
            "-cell2id", str(test_data_dir / "cell2ind.txt"),
            "-mutations", str(test_data_dir / "cell2mutation.txt"),
            "-cn_deletions", str(test_data_dir / "cell2cndeletion.txt"),
            "-cn_amplifications", str(test_data_dir / "cell2amplification.txt"),
            "-predict", str(test_data_dir / "test_data.txt"),
            "-hidden", str(hidden_dir / f"fold{fold}"),
            "-result", str(result_dir / f"fold{fold}"),
            "-load", str(PRETRAINED_MODEL(fold)),
            "-std", str(STD_FILE),
            "-cuda", "0",
            "-batchsize", "2000",
        ]
        (hidden_dir / f"fold{fold}").mkdir(parents=True, exist_ok=True)

        print(f"[{cohort_name}] fold {fold} ...")
        proc = subprocess.run(cmd, cwd=nest_vnn_dir, capture_output=True, text=True)
        print(proc.stdout.strip())
        if proc.returncode != 0:
            print(proc.stderr)
            raise RuntimeError(f"predict.py failed for {cohort_name} fold {fold} (exit {proc.returncode})")

    print(f"[{cohort_name}] done -> {result_dir}")

In [3]:
for cohort_name, test_data_dir in COHORTS.items():
    run_nest_vnn_predict(cohort_name, test_data_dir)

[McGill_cisplatin] fold 1 ...


Total number of cell lines = 29
Total number of genes = 718
Test correlation	NEST	nan
[McGill_cisplatin] fold 2 ...


Total number of cell lines = 29
Total number of genes = 718
Test correlation	NEST	nan
[McGill_cisplatin] fold 3 ...


Total number of cell lines = 29
Total number of genes = 718
Test correlation	NEST	nan
[McGill_cisplatin] fold 4 ...


Total number of cell lines = 29
Total number of genes = 718
Test correlation	NEST	nan
[McGill_cisplatin] fold 5 ...


...

Total number of cell lines = 36
Total number of genes = 718
Test correlation	NEST	nan
[UHN_carboplatin_strict] fold 2 ...


Total number of cell lines = 36
Total number of genes = 718
Test correlation	NEST	nan
[UHN_carboplatin_strict] fold 3 ...


Total number of cell lines = 36
Total number of genes = 718
Test correlation	NEST	nan
[UHN_carboplatin_strict] fold 4 ...


Total number of cell lines = 36
Total number of genes = 718
Test correlation	NEST	nan
[UHN_carboplatin_strict] fold 5 ...


Total number of cell lines = 36
Total number of genes = 718
Test correlation	NEST	nan
[UHN_carboplatin_strict] done -> /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/ml_models/nest_vnn/results/UHN_carboplatin_strict
